In [48]:
# !pip install pandas numpy scikit-learn tensorflow keras

In [49]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout



In [50]:
df = pd.read_csv('D:/self-learning/AI-ML/dataset/diabetes.csv')

In [51]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [52]:
df.shape

(768, 9)

In [53]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [54]:
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


In [55]:
y.head()

0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=13)

In [57]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((614, 8), (154, 8), (614,), (154,))

In [58]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)


In [59]:
model = Sequential()

model.add(Dense(32,activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/10


C:\Users\anmol.aman\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6075 - loss: 0.6921 - val_accuracy: 0.6429 - val_loss: 0.6592
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6564 - loss: 0.6377 - val_accuracy: 0.6948 - val_loss: 0.6140
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6824 - loss: 0.5976 - val_accuracy: 0.7143 - val_loss: 0.5780
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7052 - loss: 0.5667 - val_accuracy: 0.7273 - val_loss: 0.5532
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7231 - loss: 0.5438 - val_accuracy: 0.7273 - val_loss: 0.5338
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7362 - loss: 0.5264 - val_accuracy: 0.7403 - val_loss: 0.5182
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7476 - loss: 0.5124 - val_accuracy: 0.7597 - val_loss: 0.5066
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7606 - loss: 0.5007 - val_accuracy: 0.7662 - val_loss: 0.4984


In [60]:
#keras tuner
# !pip install -U keras-tuner

In [61]:
import keras_tuner as kt

In [62]:
def build_model(hp):
    model = Sequential()

    counter = 0
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
        if counter == 0:
            model.add(Dense(hp.Int('units_' + str(i), min_value=8, max_value=128, step=8), activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid', 'softmax', 'elu', 'selu', 'exponential', 'linear']), input_dim=8))
            model.add(Dropout(hp.Float('dropout_' + str(i), min_value=0.0, max_value=0.7, step=0.1)))
        else:
            model.add(Dense(hp.Int('units_' + str(i), min_value=8, max_value=128, step=8), activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid', 'softmax', 'elu', 'selu', 'exponential', 'linear'])))
            model.add(Dropout(hp.Float('dropout_' + str(i), min_value=0.0, max_value=0.7, step=0.1)))
        counter += 1

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd', 'adagrad', 'adadelta', 'adamax', 'nadam']), loss='binary_crossentropy', metrics=['accuracy'])
    return model

tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5)
tuner.search(X_train_scaled, y_train, epochs=10, validation_data=(X_test_scaled, y_test))


Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.6298701167106628

Best val_accuracy So Far: 0.7597402334213257
Total elapsed time: 00h 00m 30s


In [63]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

C:\Users\anmol.aman\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\anmol.aman\AppData\Roaming\Python\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamax', because it has 2 variables whereas the saved optimizer has 30 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 80)             │           720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │           216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,937 (7.57 KB)

 Trainable params: 1,937 (7.57 KB)

 Non-trainable params: 0 (0.00 B)

In [64]:
params = tuner.get_best_hyperparameters()[0].values
params

{'num_layers': 6,
 'units_0': 8,
 'activation_0': 'linear',
 'dropout_0': 0.0,
 'optimizer': 'adamax',
 'units_1': 80,
 'activation_1': 'relu',
 'dropout_1': 0.4,
 'units_2': 8,
 'activation_2': 'elu',
 'dropout_2': 0.0,
 'units_3': 24,
 'activation_3': 'tanh',
 'dropout_3': 0.0,
 'units_4': 8,
 'activation_4': 'relu',
 'dropout_4': 0.0,
 'units_5': 8,
 'activation_5': 'relu',
 'dropout_5': 0.0}

In [65]:
best_model.fit(X_train_scaled, y_train, epochs=100, initial_epoch=10, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7410 - loss: 0.5484 - val_accuracy: 0.7468 - val_loss: 0.5445
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7378 - loss: 0.5401 - val_accuracy: 0.7532 - val_loss: 0.5362
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7492 - loss: 0.5218 - val_accuracy: 0.7532 - val_loss: 0.5305
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7410 - loss: 0.5250 - val_accuracy: 0.7468 - val_loss: 0.5255
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7638 - loss: 0.5084 - val_accuracy: 0.7403 - val_loss: 0.5203
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7508 - loss: 0.5147 - val_accuracy: 0.7403 - val_loss: 0.5174
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7606 - loss: 0.5045 - val_accuracy: 0.7532 - val_loss: 0.5132
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7590 - loss: 0.5017 - val_accurac